# 🚚⚠️📊 Прогнозирование риска задержки доставки CargaPronto: сегментация клиентов и бинарная классификация заказов

Автор: Якунин Михаил

Дата: "__" _______ 2026 г.

***По этой ссылке можно посмотреть проект и скачать последнюю версию: https://github.com/MEYakunin/Sprint18_Delivery_Delay_Prediction.git***

## Цель проекта

*Цель проекта* — построить для логистической компании CargaPronto модель бинарной классификации, которая по данным о заказе и профиле клиента будет предсказывать риск задержки доставки (`late_delivery_risk`) ещё в момент оформления заказа. 

*Конечный продукт* — система поддержки принятия решений для диспетчеров: при высоком риске опоздания (>70%) система должна выдавать предупреждение и рекомендовать сменить приоритет доставки, что позволит компании активно управлять ситуацией вместо того, чтобы констатировать факт нарушения SLA.

*Ключевая задача исследования* — проверить гипотезу транспортного департамента о «неоднородности» клиентской базы: подтвердить, что риск задержки определяется не только параметрами самого заказа, но и географическим положением клиента, а также его поведенческим профилем (частота заказов, сумма покупок, доля возвратов, чувствительность к скидкам). Для этого клиенты будут сегментированы с помощью кластеризации K-Means++ на логистические зоны (`geo_cluster`) и поведенческие сегменты (`beh_cluster`), после чего полученные признаки добавляются в обучающие данные моделей.

*Целевой показатель качества* — ROC-AUC > 0.75 на тестовой выборке: метрика оценивает способность модели ранжировать заказы по степени риска, что позволит диспетчерам точечно выделять доставки, требующие повышенного внимания.

## Задача проекта

*Задача проекта* — разработать модель бинарной классификации, которая в момент оформления заказа предсказывает целевую переменную `late_delivery_risk`: 1 — заказ доставлен с задержкой, 0 — доставлен вовремя. Входными данными служат параметры самого заказа (`order_id`, `customer_id`, `order_date`, `shipping_mode`, `category_name`, `item_price`) и профиль клиента из справочника (`customer_lat`, `customer_lon`, `recency`, `total_orders`, `total_sales`, `return_rate` и `avg_discount`).

*Инструменты моделирования* — логистическая регрессия (быстрая и прозрачная, задаёт «нижнюю планку» качества) и `CatBoostClassifier` (градиентный бустинг, способный находить сложные закономерности). Метрика качества — `ROC-AUC`: она оценивает способность модели ранжировать заказы по степени риска, что позволяет диспетчерам точечно выделять доставки, требующие повышенного внимания. Удовлетворительный результат — `ROC-AUC` > 0.75 на тестовой выборке.

*План работы:*
1. Загрузка датасетов `ds_s18_customers.csv` и `ds_s18_orders.csv`, технический аудит, устранение полных дубликатов, проверка целостности данных, подготовка временных признаков `order_month`, `order_weekday`, `order_hour`.
2. EDA: проверка баланса классов, географический аудит и удаление аномалий координат, анализ зависимостей признаков.
3. Разделение заказов на обучающую, валидационную и тестовую выборки в соотношении 60:20:20 с помощью `GroupShuffleSplit` (клиенты не должны пересекаться между выборками).
4. Обучение базовых моделей (логистическая регрессия и `CatBoostClassifier`) на признаках заказа — фиксация точки отсчёта качества.
5. Кластеризация клиентов: геокластеры (`geo_cluster`) по координатам и поведенческие сегменты (`beh_cluster`) по RFM-признакам, с выбором числа кластеров методом локтя и защитой от утечки данных.
6. Обучение финальных моделей на расширенном наборе признаков и оценка ROC-AUC на валидационной выборке.
7. Тестирование лучшей модели на тестовой выборке: итоговый ROC-AUC, матрица ошибок, анализ важности признаков.
8. Выводы и бизнес-рекомендации для CargaPronto.

## Описание данных

В проекте используются два датасета, выгруженные из ERP-системы компании CargaPronto.

1. `ds_s18_customers.csv` — профили уникальных клиентов

Справочник содержит агрегированную информацию о каждом клиенте: географическое положение и поведенческие показатели, собранные по методу RFM.

| Признак | Описание |
| :--- | :--- |
| `customer_id` | Уникальный номер клиента в системе ERP; ключ для связи с таблицей заказов |
| `customer_lat` | Широта местонахождения клиента |
| `customer_lon` | Долгота местонахождения клиента |
| `recency` | Количество дней, прошедших с момента последнего заказа до точки отсчёта (Recency) |
| `total_orders` | Общее количество совершённых заказов (Frequency) |
| `total_sales` | Суммарная выручка, которую клиент принёс компании за всё время (Monetary) |
| `avg_discount` | Средний размер скидки, которую получает клиент; маркер чувствительности к акциям |
| `return_rate` | Доля проблемных заказов (возвраты и отмены); ключевой показатель нагрузки на курьеров |

2. `ds_s18_orders.csv` — заказы клиентов

Таблица событий, где зафиксирован каждый факт продажи и итог доставки; здесь находится целевая переменная.

| Признак | Описание |
| :--- | :--- |
| `order_id` | Уникальный номер заказа |
| `customer_id` | Идентификатор клиента; ключ для связи с профилем из справочника |
| `order_date` | Дата и время совершения покупки |
| `category_name` | Категория заказанного товара |
| `item_price` | Стоимость товара в заказе |
| `shipping_mode` | Режим доставки: Standard Class, First Class, Second Class, Same Day |
| `late_delivery_risk` | Целевая переменная: 1 — заказ доставлен с задержкой; 0 — заказ доставлен вовремя |

## Содержание <a id="content"></a>

* [1. Импорт библиотек, выполнение базовых настроек](#1)
* [2. Загрузка данных, первичное знакомство](#2)
  * [2.1. Технический аудит данных](#2_1)
  * [2.2. Устранение дубликатов и проверка целостности (ID-Check)](#2_2)
  * [2.3. Подготовка признаков даты и времени](#2_3)
* [3. EDA](#3)
  * [3.1. Проверка баланса классов](#3_1)
  * [3.2. Географический аудит](#3_2)
  * [3.3. Удаление аномалий](#3_3)
  * [3.4. Анализ зависимостей между признаками](#3_4)
* [4. Разделение на выборки](#4)
* [5. Обучение базовой модели](#5)
  * [5.1. Логистическая регрессия](#5_1)
  * [5.2. CatBoostClassifier](#5_2)
  * [5.3. Оценка качества на валидационной выборке](#5_3)
* [6. Кластеризация по признакам, связанным с местоположением](#6)
  * [6.1. Подготовка признаков и защита от утечки данных](#6_1)
  * [6.2. Выбор числа кластеров методом локтя](#6_2)
  * [6.3. Визуализация кластеров и центроидов](#6_3)
* [7. Кластеризация по признакам профилей клиентов](#7)
  * [7.1. Подготовка векторов признаков и предобработка](#7_1)
  * [7.2. Выбор числа кластеров методом локтя](#7_2)
  * [7.3. Анализ полученных сегментов](#7_3)
  * [7.4. Визуализация кластеров с помощью t-SNE](#7_4)
* [8. Обучение модели на новых признаках](#8)
  * [8.1. Добавление `geo_cluster` и `beh_cluster` в данные](#8_1)
  * [8.2. Обучение финальных моделей](#8_2)
  * [8.3. Оценка ROC-AUC на валидационной выборке](#8_3)
* [9. Дополнительное задание — подбор лучшего количества кластеров](#9)
* [10. Тестирование лучшей модели](#10)
  * [10.1. Предсказание на тестовой выборке и итоговый ROC-AUC](#10_1)
  * [10.2. Матрица ошибок](#10_2)
  * [10.3. Анализ важности признаков](#10_3)
* [11. Выводы и рекомендации](#11)

## [1. Импорт библиотек, выполнение базовых настроек](#content)<a id="1"></a>

Загрузите все необходимые для выполнения проекта библиотеки и другие компоненты.



In [ ]:
#%pip install -r requirements.txt

## [2. Загрузка данных, первичное знакомство](#content)<a id="2"></a>

* Загрузите датасеты `ds_s18_customers.csv` и `ds_s18_orders.csv`.
  * Путь к первому файлу — `'/datasets/ds_s18_customers.csv'`.
  * Путь ко второму файлу — `'/datasets/ds_s18_orders.csv'`.
* Проведите технический аудит данных.
* Выявите и устраните полные дубликаты строк.
* Выполните проверку целостности данных (ID-Check).
* Подготовьте признаки, связанные с датой и временем.
* Создайте дополнительные колонки `order_month`, `order_weekday`, `order_hour`.




### [2.1. Технический аудит данных](#content)<a id="2_1"></a>

### [2.2. Устранение дубликатов и проверка целостности (ID-Check)](#content)<a id="2_2"></a>

### [2.3. Подготовка признаков даты и времени](#content)<a id="2_3"></a>

## [3. EDA](#content)<a id="3"></a>

На этом этапе ваша цель — исследовать структуру данных и подтвердить гипотезы транспортного департамента о «неоднородности» базы. Кроме того, вам предстоит изучить и удалить географические аномалии.

1. Проверьте баланс классов.
2. Выполните географический аудит.
3. Удалите аномалии.
4. Проанализируйте зависимости между признаками.



### [3.1. Проверка баланса классов](#content)<a id="3_1"></a>

### [3.2. Географический аудит](#content)<a id="3_2"></a>

### [3.3. Удаление аномалий](#content)<a id="3_3"></a>

### [3.4. Анализ зависимостей между признаками](#content)<a id="3_4"></a>

## [4. Разделение на выборки](#content)<a id="4"></a>

Разделите датафрейм с заказами на выборки. В этом проекте вам предстоит  использовать «классическое» разделение на три выборки: обучающую, валидационную и тестовую. Использовать для этого нужно не `train_test_split`, а `GroupShuffleSplit`. Он делит данные так, чтобы группы, то есть клиенты, не пересекались.

Ниже приведён пример того, как отделить часть данных для обучения, используя `customer_id` как ключ для группировки.

```python
from sklearn.model_selection import GroupShuffleSplit

# 1. Выбираем колонку, по которой будем группировать (наши "группы")
groups = df_orders['customer_id']

# 2. Инициализируем сплиттер (например, отделим 20% клиентов для теста)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)

# 3. Получаем индексы для разделения
# Метод split возвращает генератор, поэтому используем next()
train_idx, test_idx = next(gss.split(df_orders, groups=groups))

# 4. Формируем выборки
df_train = df_orders.iloc[train_idx]
df_test = df_orders.iloc[test_idx]

```


1. Разделите данные из датафрейма с заказами  на три выборки  в соотношении 60:20:20.


2. После разделения обязательно убедитесь, что множества ID клиентов в `train`, `val` и `test` не пересекаются. Если один и тот же клиент попал в разные выборки, то разделение выполнено неверно.

## [5. Обучение базовой модели](#content)<a id="5"></a>

Прежде чем проверять гипотезу о сегментации и создавать при помощи кластеризации новые признаки, необходимо зафиксировать точку отсчёта. Вам нужно понять, какое качество прогноза задержек обеспечивают стандартные модели, используя только базовую информацию о самом заказе.

1. Выберите нужные признаки — используйте колонки таблицы `orders` (`shipping_mode`, `category_name`, `item_price`, `order_hour`,  `order_weekday` и `order_month`).

2. Обучите логистическую регрессию и `CatBoostClassifier` и оцените их качество.

3. Оцените качество обеих моделей на валидационной выборке.

Напоминаем, что на этапе построения базовой модели глубокий подбор гиперпараметров необязателен.

**Дополнительное задание.** Если вы чувствуете в себе силы, вы можете попробовать базовую оптимизацию, но помните: главная прибавка к качеству ожидается от новых признаков, которые вы создадите в следующих разделах.






### [5.1. Логистическая регрессия](#content)<a id="5_1"></a>

### [5.2. CatBoostClassifier](#content)<a id="5_2"></a>

### [5.3. Оценка качества на валидационной выборке](#content)<a id="5_3"></a>

## [6. Кластеризация по признакам, связанным с местоположением](#content)<a id="6"></a>



На этом этапе нужно превратить исходные координаты `customer_lat` и `customer_lon` в полезный для модели признак — логистическую зону.

**Ваши действия:**

1. Проведите корректную подготовку признаков.
2. Продумайте защиту от утечки данных.
3. Используйте метод локтя, чтобы найти математически обоснованную границу количества групп.
4. Постройте график с полученными кластерами и отметьте центроиды.

### [6.1. Подготовка признаков и защита от утечки данных](#content)<a id="6_1"></a>

### [6.2. Выбор числа кластеров методом локтя](#content)<a id="6_2"></a>

### [6.3. Визуализация кластеров и центроидов](#content)<a id="6_3"></a>

## [7. Кластеризация по признакам профилей клиентов](#content)<a id="7"></a>





1. Подготовьте векторы признаков — используйте пять ключевых показателей из датасета `df_customers.csv`: `recency`, `total_orders`, `total_sales`, `return_rate` и `avg_discount`.
2. Выполните предобработку данных и обеспечьте защиту от утечек.
3. Используйте метод локтя, чтобы найти математически обоснованную границу количества групп.
4. Проанализируйте полученные кластеры: дайте статистическую характеристику и опишите самые яркие группы.
5. Визуализируйте положение кластеров с помощью t-SNE.



### [7.1. Подготовка векторов признаков и предобработка](#content)<a id="7_1"></a>

### [7.2. Выбор числа кластеров методом локтя](#content)<a id="7_2"></a>

### [7.3. Анализ полученных сегментов](#content)<a id="7_3"></a>

### [7.4. Визуализация кластеров с помощью t-SNE](#content)<a id="7_4"></a>

## [8. Обучение модели на новых признаках](#content)<a id="8"></a>

1. Добавьте результаты кластеризаций в данные для обучения моделей.
2. Обучите финальные версии логистической регрессии и `CatBoostClassifier` на расширенном наборе данных.
3. Рассчитайте итоговые значения метрики ROC−AUC на валидационной выборке.
   

### [8.1. Добавление `geo_cluster` и `beh_cluster` в данные](#content)<a id="8_1"></a>

### [8.2. Обучение финальных моделей](#content)<a id="8_2"></a>

### [8.3. Оценка ROC-AUC на валидационной выборке](#content)<a id="8_3"></a>

## [9. Дополнительное задание — подбор лучшего количества кластеров](#content)<a id="9"></a>

Рассмотрите количество кластеров как гиперпараметр всей системы и найти ту «степень детализации», которая даст максимальный прирост ROC-AUC.

Вы можете взять значения из списка или выбрать свои:

```python
geo_k_range = [4, 8, 12, 20]
rfm_k_range = [6, 12, 20, 30]
```



## [10. Тестирование лучшей модели](#content)<a id="10"></a>

На этом этапе вы должны убедиться, что выбранная модель сохраняет высокое качество на данных, которые она никогда не видела, и понять, какие факторы стали решающими для прогноза.

1. Выполните предсказание на тестовой выборке для лучшей модели. Рассчитайте итоговый ROC-AUC и сравните его с целевым показателем 0.75.
2. Постройте матрицу ошибок. Определите, какой тип ошибок совершает модель чаще: пропускает ли она реальные задержки или слишком часто выдаёт ложную тревогу?
3. Визуализируйте важность признаков вашей лучшей модели.
4. Проанализируйте позиции созданных вами признаков `geo_cluster` и `beh_cluster` в общем рейтинге. Стали ли они ключевыми факторами для предсказаний модели для модели или базовые параметры заказа (цена, время) остались приоритетными?

### [10.1. Предсказание на тестовой выборке и итоговый ROC-AUC](#content)<a id="10_1"></a>

### [10.2. Матрица ошибок](#content)<a id="10_2"></a>

### [10.3. Анализ важности признаков](#content)<a id="10_3"></a>

## [11. Выводы и рекомендации](#content)<a id="11"></a>

Что нужно зафиксировать:

* **Результаты моделирования.** Укажите итоговое значение ROC-AUC на тестовой выборке. Удалось ли вам достичь целевого показателя? Насколько эффективно итоговая модель справляется с выявлением задержек по сравнению с базовыми?
* **Эффективность сегментации.** Сформулируйте вывод о полезности кластеризации. Подтвердилась ли гипотеза о том, что профиль клиента и его географическое положение влияют на риск задержки? Какие именно кластеры — географические или поведенческие — оказались более информативными для модели? Добавьте в раздел визуализации, которые вы получили, работая над проектом.
* **Технический инсайт.** Если вы экспериментировали с разным количеством  кластеров как с гиперпараметром, то укажите, какое количество кластеров K оказалось оптимальным. Кратко поясните, почему слишком большое K может вредить качеству прогноза.
* **Бизнес-рекомендации.** Предложите, как CargaPronto может использовать вашу модель в реальных операциях.